# SONARA final evaluation

This is the notebook whose numbers go into the paper. It evaluates, **once**, the setup chosen in Run 2 on data that no earlier experiment has used.

**The system model was declared before this notebook was written** (`results/run2_second_pass/FINDINGS.md`). Its training settings come from the hyperparameter tuning notebook (`SONARA_Hyperparameter_Tuning.ipynb`), which must be run first; this notebook reads its decision from `chosen_system_model_config.json` and stops if the file is missing.

| Role | Setup | Model and settings |
| :- | :- | :- |
| **System model** | B8 | MobileNetV3-Small, ImageNet weights, no normalization, loudness augmentation; training settings from the tuning notebook |
| Runner-up, for comparison | C2 | EfficientNet-B0, ImageNet weights, training recipe, no normalization |
| Baseline, for comparison | A0 | CNN–BiLSTM with the Run 1 settings |

## How the notebook is ordered

| Stage | What happens | Data it uses |
| :- | :- | :- |
| A. Train | Each setup is trained on 5 seeds. Each run picks two decision thresholds on its own validation clips: the highest threshold that still catches 90% of Distress clips, and the highest that catches 95%. | Development data only (85% of SmartEars) |
| B. Freeze | All trained models and both thresholds are listed in `frozen_models.json`, with a fingerprint of every weight file. | — |
| C. Open the test data | Every frozen model scores the **locked test set** (871 SmartEars clips, id `bf636617`) and **PVSD** (346 files). Predictions are saved; each opening is logged. | Locked test set and PVSD, **once** |
| D. Report | Tables and figures are computed from the saved predictions. Nothing is retrained or re-tuned. | Saved predictions |
| E. Zone alerts | The alert rule is chosen on validation predictions, then reported on test predictions. | Validation, then test predictions |
| F. Export | The system model is saved for the Raspberry Pi gateway. | — |

**Rules.** Do not change any setting after Stage C has run. If something must change, say so in the paper and report both results; the log in `test_opened_log.json` records every opening of the test data.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow Google Drive access. Expect about 3 hours on a T4. Finished training runs are skipped on rerun.

## 1. Setup and configuration

In [ ]:
import os, re, copy, glob, hashlib, json, math, random, time
from datetime import datetime, timezone
from math import gcd

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import resample_poly

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchvision
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, LambdaLR

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, roc_curve, confusion_matrix)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
if device.type == "cpu":
    print("WARNING: no GPU. Switch the Colab runtime to a T4 GPU before training.")

In [ ]:
# ---- Data and outputs ----------------------------------------------------
REPO_URL = "https://github.com/tagalogclark123-cmyk/SONARA-GITHUB.git"
REPO_DIR = "/content/SONARA-GITHUB"
SAVE_TO_DRIVE = True
OUT_DIR = "/content/drive/MyDrive/SONARA/final_evaluation" if SAVE_TO_DRIVE else "/content/final_evaluation"
DATASETS = [
    {"name": "SmartEars", "root": "Smartears", "label_map": {"Healthy": "Normal", "None": "Noise", "Sick": "Distress"}},
    {"name": "PVSD", "root": "PVSD", "label_map": {"Healthy": "Normal", "Noise": "Noise", "Unhealthy": "Distress"}},
]

# ---- Protocol (same as Run 2) -----------------------------------------------
LOCK_SEED = 2026
EXPECTED_LOCK_ID = "bf636617"
FINAL_SEEDS = [7, 21, 42, 77, 99]
PRIMARY_SEED = 42             # the run used for figures and for the exported gateway model
GROUP_BY = "recording"
TARGET_RECALLS = [0.90, 0.95]  # operating points: validation distress recall each threshold must reach
PRIMARY_TARGET = 0.90         # the operating point used for the main results and the gateway
LEVEL_OFFSETS_DB = [-20, -10, -6, 0, 6, 10, 20]
N_BOOTSTRAP = 1000            # resamples of test recordings for the 95% confidence intervals

# ---- Input and augmentation (same as Run 2) ---------------------------------
SAMPLE_RATE = 16000
CROP_SEC = 5.0
N_FFT, HOP_LENGTH, N_MELS = 400, 160, 64
MAX_EXTERNAL_WINDOWS = 50     # cap on 5-s windows scored per PVSD file (evenly spaced if more)

# ---- The three setups, exactly as in Run 2 ----------------------------------
RUN1 = {"norm": "none", "pretrained": False, "optimizer": "adam", "lr": 1e-4, "weight_decay": 1e-4,
        "schedule": "plateau", "max_epochs": 50, "patience": 8, "mixup": 0.0,
        "three_class": False, "gain_prob": 0.2, "gain_db": 6.0}
RECIPE = {"optimizer": "adamw", "lr": 3e-4, "weight_decay": 1e-2, "schedule": "cosine",
          "max_epochs": 40, "patience": 10, "mixup": 0.2}
SETUPS = {
    "B8_mobilenet_pre_recipe_gain": {"model": "mobilenetv3_small", "role": "system model", **RUN1, **RECIPE,
                                     "pretrained": True, "gain_prob": 1.0, "gain_db": 12.0},
    "C2_efficientnet_pre_recipe":   {"model": "efficientnet_b0", "role": "runner-up", **RUN1, **RECIPE, "pretrained": True},
    "A0_cnn_bilstm_run1":           {"model": "cnn_bilstm", "role": "Run 1 baseline", **RUN1},
}
# Settings Run 2 did not vary, filled in for every setup unless the setup or the tuning result says otherwise.
TRAINING_DEFAULTS = {"batch_size": 16, "label_smoothing": 0.02, "warmup": 2, "shift_sec": 0.5, "shift_prob": 0.5,
                     "time_mask_prob": 0.4, "freq_mask_prob": 0.4}
for name, cfg in SETUPS.items():
    for k, v in TRAINING_DEFAULTS.items():
        cfg.setdefault(k, v)
    cfg.setdefault("head_dropout", 0.3 if cfg["model"] == "cnn_bilstm" else 0.2)   # V5 value; torchvision default
SYSTEM_MODEL = "B8_mobilenet_pre_recipe_gain"

# ---- Training settings of the system model, from the tuning notebook --------
TUNED_CONFIG_PATH = "/content/drive/MyDrive/SONARA/hyperparameter_tuning/chosen_system_model_config.json"
TUNABLE = ["optimizer", "lr", "weight_decay", "batch_size", "schedule", "warmup", "max_epochs", "head_dropout",
           "label_smoothing", "mixup", "gain_db", "gain_prob", "shift_sec", "shift_prob", "time_mask_prob", "freq_mask_prob"]
LABELS = {"B8_mobilenet_pre_recipe_gain": "MobileNetV3-Small (B8, system model)",
          "C2_efficientnet_pre_recipe": "EfficientNet-B0 (C2)",
          "A0_cnn_bilstm_run1": "CNN-BiLSTM, Run 1 settings (A0)"}

# ---- Zone alert simulation (Part E), fixed before the test data is opened ---
CLIP_INTERVAL_MIN = 0.5       # one 5-s clip per zone every 30 s
ALERT_WINDOWS = [10, 20, 40, 60]                   # number of recent clips the alert looks at
ALERT_RATES = [0.2, 0.3, 0.4, 0.5, 0.6]            # share of them that must be flagged
MAX_FALSE_ALERT_HOURS_PER_WEEK = 1.0               # hours per week in which a healthy zone shows an alert
DESIGN_SHARE = 0.3            # share of a sick zone's clips that contain distress, used to choose the rule
OUTBREAK_SHARES = [0.1, 0.2, 0.3, 0.5]
ZONE_REPS = 200               # simulated weeks / outbreaks per setting

QUICK_RUN = None              # e.g. 40 = 40 clips per class, 2 epochs, 2 seeds (code check only)

CLASS_NAMES = ["Non-distress", "Distress"]
SOURCE_INDEX = {"Normal": 0, "Distress": 1, "Noise": 2}
CROP_SAMPLES = int(round(CROP_SEC * SAMPLE_RATE))
N_FRAMES = CROP_SAMPLES // HOP_LENGTH
NUM_WORKERS = 2 if os.name == "posix" else 0
if QUICK_RUN:
    FINAL_SEEDS, N_BOOTSTRAP, ZONE_REPS = FINAL_SEEDS[:2], 50, 10
    PRIMARY_SEED = FINAL_SEEDS[0]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if os.path.exists(TUNED_CONFIG_PATH):
    tuned = json.load(open(TUNED_CONFIG_PATH))
    SETUPS[SYSTEM_MODEL].update({k: tuned["config"][k] for k in TUNABLE if k in tuned["config"]})
    print("Tuning decision:", tuned["decision"])
elif not QUICK_RUN:
    raise FileNotFoundError(f"{TUNED_CONFIG_PATH} not found. Run SONARA_Hyperparameter_Tuning.ipynb first.")
RUN_DIR = os.path.join(OUT_DIR, "runs")
TEST_DIR = os.path.join(OUT_DIR, "test_predictions")
for d in (RUN_DIR, TEST_DIR, os.path.join(OUT_DIR, "export")):
    os.makedirs(d, exist_ok=True)
print("Outputs:", OUT_DIR)
print(pd.DataFrame(SETUPS).T[["model", "role"] + TUNABLE].to_string())

In [ ]:
# Download the repository (code + SmartEars and PVSD audio). Takes several minutes the first time.
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
print("Repo folder:", REPO_DIR, "| exists:", os.path.isdir(REPO_DIR))

## 2. Data

SmartEars and PVSD are loaded as in Run 1 and Run 2: 16 kHz mono, binary labels (0 = Non-distress, 1 = Distress). PVSD is only scored in Stage C; it is never trained on and never used to choose anything.

In [ ]:
SMARTEARS_NAME = re.compile(r"^(rpi\d+)-(\d{8})(\d{6})(?:-(\d+))?$")
rows = []
for ds in DATASETS:
    root = os.path.join(REPO_DIR, ds["root"])
    for folder, source in ds["label_map"].items():
        for path in sorted(glob.glob(os.path.join(root, folder, "*.wav"))):
            stem = os.path.splitext(os.path.basename(path))[0]
            m = SMARTEARS_NAME.match(stem)
            recording = f"{m.group(1)}-{m.group(2)}{m.group(3)}" if m else stem
            rows.append({"path": path, "dataset": ds["name"], "file": f"{folder}/{os.path.basename(path)}",
                         "original_label": folder, "source": source, "label": int(source == "Distress"),
                         "recording": recording})
manifest = pd.DataFrame(rows)
if QUICK_RUN:
    manifest = manifest.groupby(["dataset", "source"]).head(QUICK_RUN).reset_index(drop=True)
print(manifest.groupby(["dataset", "original_label", "source"]).size().to_string())


def load_audio(path):
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != SAMPLE_RATE:
        g = gcd(int(sr), SAMPLE_RATE)
        wav = resample_poly(wav, SAMPLE_RATE // g, sr // g).astype(np.float32)
    return wav


t0 = time.time()
clips = []
for i, path in enumerate(manifest["path"]):
    clips.append(np.clip(load_audio(path) * 32767.0, -32768, 32767).astype(np.int16))
    if (i + 1) % 1000 == 0:
        print(f"  loaded {i + 1}/{len(manifest)} clips ({time.time() - t0:.0f}s)")
manifest["seconds"] = [len(c) / SAMPLE_RATE for c in clips]
print(f"Loaded {len(clips)} clips in {time.time() - t0:.0f}s")

## 3. Locked test set and development splits

The locked test set is recomputed exactly as in Run 2 and must have id `bf636617`. Each seed then splits the development data into training (70% of SmartEars) and validation (15%), grouped by source recording. Seeds 7, 21 and 42 reproduce the Run 2 splits; 77 and 99 are new.

In [ ]:
def grouped_folds(indices, n_splits, seed):
    df = manifest.loc[indices]
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_of = np.zeros(len(df), dtype=int)
    for fold, (_, idx) in enumerate(sgkf.split(df, df["source"], df[GROUP_BY])):
        fold_of[idx] = fold
    return fold_of


smartears_idx = manifest.index[manifest["dataset"] == "SmartEars"].to_numpy()
pvsd_idx = manifest.index[manifest["dataset"] == "PVSD"].to_numpy()
fold_of = grouped_folds(smartears_idx, 20, LOCK_SEED)
LOCKED = smartears_idx[fold_of >= 17]
dev_idx = smartears_idx[fold_of < 17]
assert not set(manifest.loc[LOCKED, GROUP_BY]) & set(manifest.loc[dev_idx, GROUP_BY]), "group leakage"
lock_id = hashlib.md5("\n".join(sorted(manifest.loc[LOCKED, "file"])).encode()).hexdigest()[:8]
print(f"Locked test set: {len(LOCKED)} clips, id {lock_id} | development set: {len(dev_idx)} clips | PVSD: {len(pvsd_idx)} files")
if not QUICK_RUN:
    assert lock_id == EXPECTED_LOCK_ID, f"Locked test set id {lock_id} differs from Run 2 ({EXPECTED_LOCK_ID}). Stop."
DEV = set(dev_idx.tolist())


def make_split(seed):
    fold_of = grouped_folds(dev_idx, 17, seed)
    split = {"train": dev_idx[fold_of < 14], "val": dev_idx[fold_of >= 14]}
    assert not set(manifest.loc[split["train"], GROUP_BY]) & set(manifest.loc[split["val"], GROUP_BY]), "group leakage"
    return split


for s in FINAL_SEEDS:
    sp = make_split(s)
    print(f"seed {s}: train {len(sp['train'])} {np.bincount(manifest.loc[sp['train'], 'label']).tolist()} | "
          f"val {len(sp['val'])} {np.bincount(manifest.loc[sp['val'], 'label']).tolist()}")

## 4. Features, augmentation and models

Identical to Run 2. Features are mel power in dB plus delta and delta-delta, 3 × 64 × 500. Short clips are padded on the features (dB channel with the clip's own minimum, deltas with 0).

In [ ]:
class LogMelDeltas(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                                                        n_mels=N_MELS, center=True, power=2.0)
        self.to_db = torchaudio.transforms.AmplitudeToDB(stype="power")
        self.deltas = torchaudio.transforms.ComputeDeltas()

    def features(self, wav):
        db = self.to_db(self.mel(wav))
        d1 = self.deltas(db)
        return torch.stack([db, d1, self.deltas(d1)], dim=1)

    def forward(self, wav, lengths):
        x = self.features(wav)[..., :N_FRAMES]
        for i in torch.nonzero(lengths < CROP_SAMPLES).flatten().tolist():
            f = self.features(wav[i:i + 1, :max(int(lengths[i]), N_FFT)])[0]
            t = min(f.shape[-1], N_FRAMES)
            x[i, 0] = f[0].min()
            x[i, 1:] = 0.0
            x[i, :, :, :t] = f[:, :, :t]
        return x


featurizer = LogMelDeltas().to(device).eval()


def normalize(x, mode):
    if mode == "per_clip":
        return (x - x.mean(dim=(2, 3), keepdim=True)) / (x.std(dim=(2, 3), keepdim=True) + 1e-5)
    return x


def augment_features(x, cfg):
    _, _, n_freq, n_time = x.shape
    max_shift = int(cfg["shift_sec"] * SAMPLE_RATE / HOP_LENGTH)
    for i in range(x.size(0)):
        if max_shift and random.random() < cfg["shift_prob"]:
            x[i] = torch.roll(x[i], random.randint(-max_shift, max_shift), dims=2)
        ch_mean = x[i].mean(dim=(1, 2), keepdim=True)
        if random.random() < cfg["time_mask_prob"]:
            width = random.randint(5, n_time // 8)
            t0 = random.randint(0, n_time - width)
            x[i, :, :, t0:t0 + width] = ch_mean
        if random.random() < cfg["freq_mask_prob"]:
            height = random.randint(3, n_freq // 8)
            f0 = random.randint(0, n_freq - height)
            x[i, :, f0:f0 + height, :] = ch_mean
        if random.random() < cfg["gain_prob"]:
            x[i, 0] += random.uniform(-cfg["gain_db"], cfg["gain_db"])
    return x


def forward_features(wav, n, cfg, training=False, level_offset_db=0.0):
    with torch.no_grad():
        x = featurizer(wav, n)
        if level_offset_db:
            x[:, 0] += level_offset_db
        if training:
            x = augment_features(x, cfg)
        return normalize(x, cfg["norm"])


class ClipDataset(Dataset):
    """5-s clips from `indices`. `allowed` lists the clips this stage may touch."""

    def __init__(self, indices, allowed):
        self.indices = np.asarray(indices)
        assert set(self.indices.tolist()) <= allowed, "this stage requested clips it may not use"
        self.labels = manifest["label"].to_numpy()
        self.sources = manifest["source"].map(SOURCE_INDEX).to_numpy()

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        seg = clips[idx][:CROP_SAMPLES].astype(np.float32) / 32768.0
        n = len(seg)
        if n < CROP_SAMPLES:
            seg = np.pad(seg, (0, CROP_SAMPLES - n))
        return torch.from_numpy(seg), int(self.labels[idx]), int(self.sources[idx]), n


def make_loader(indices, train, seed, allowed=DEV, batch_size=64):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ClipDataset(indices, allowed), batch_size=batch_size, shuffle=train, num_workers=NUM_WORKERS,
                      generator=g, pin_memory=(device.type == "cuda"))


class CNN_BiLSTM(nn.Module):
    def __init__(self, n_mels=64, num_classes=2, cnn_channels=64, lstm_hidden=128, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(cnn_channels)
        self.conv2 = nn.Conv2d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(cnn_channels)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.lstm = nn.LSTM(input_size=cnn_channels * (n_mels // 4), hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        b, c, fq, t = x.shape
        x = x.permute(0, 3, 1, 2).reshape(b, t, c * fq)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat([h_n[-2], h_n[-1]], dim=1)))


def build_model(name, pretrained, head_dropout):
    weights = "IMAGENET1K_V1" if pretrained else None
    if name == "cnn_bilstm":
        return CNN_BiLSTM(dropout=head_dropout)
    if name == "mobilenetv3_small":
        m = torchvision.models.mobilenet_v3_small(weights=weights)
    elif name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=weights)
    else:
        raise ValueError(name)
    m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, 2)
    for layer in m.classifier:
        if isinstance(layer, nn.Dropout):
            layer.p = head_dropout
    return m


def count_params(model):
    return sum(p.numel() for p in model.parameters())

## 5. Stage A: train every setup on every seed

Training is identical to Run 2: weighted cross-entropy with label smoothing 0.02, best checkpoint at the lowest validation loss. After training, each run picks its decision thresholds (90% and 95% validation recall) on its own validation clips and saves its weights. Finished runs are skipped on rerun.

In [ ]:
def run_epoch(model, loader, cfg, criterion, optimizer=None, level_offset_db=0.0):
    training = optimizer is not None
    model.train(training)
    total_loss, total, probs, labels = 0.0, 0, [], []
    with torch.set_grad_enabled(training):
        for wav, y, _, n in loader:
            wav, y = wav.to(device), y.to(device)
            x = forward_features(wav, n, cfg, training, level_offset_db)
            if training and cfg["mixup"] > 0:
                lam = float(np.random.beta(cfg["mixup"], cfg["mixup"]))
                perm = torch.randperm(x.size(0), device=device)
                logits = model(lam * x + (1 - lam) * x[perm])
                loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y[perm])
            else:
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            else:
                probs.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
                labels.append(y.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            total += y.size(0)
    if training:
        return total_loss / total, None, None
    return total_loss / total, np.concatenate(probs), np.concatenate(labels)


def make_criterion(train_idx, cfg):
    counts = np.bincount(manifest.loc[train_idx, "label"], minlength=2)
    weights = torch.tensor(counts.mean() / counts, dtype=torch.float32, device=device)
    return nn.CrossEntropyLoss(weight=weights, label_smoothing=cfg["label_smoothing"])


def train_setup(name, seed, split):
    cfg = SETUPS[name]
    max_epochs = 2 if QUICK_RUN else cfg["max_epochs"]
    set_seed(seed)
    model = build_model(cfg["model"], cfg["pretrained"], cfg["head_dropout"]).to(device)
    criterion = make_criterion(split["train"], cfg)
    if cfg["optimizer"] == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    if cfg["schedule"] == "cosine":
        warm = cfg["warmup"]
        scheduler = LambdaLR(optimizer, lambda e: (e + 1) / warm if e < warm else
                             max(0.02, 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, max_epochs - warm)))))
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6)
    train_loader = make_loader(split["train"], True, seed, batch_size=cfg["batch_size"])
    val_loader = make_loader(split["val"], False, seed)
    history, best, stale = [], {"loss": float("inf")}, 0
    for epoch in range(1, max_epochs + 1):
        train_loss, _, _ = run_epoch(model, train_loader, cfg, criterion, optimizer)
        val_loss, p_val, y_val = run_epoch(model, val_loader, cfg, criterion)
        scheduler.step(val_loss) if cfg["schedule"] == "plateau" else scheduler.step()
        improved = val_loss < best["loss"] - 1e-4
        if improved:
            best, stale = {"loss": val_loss, "epoch": epoch, "p_val": p_val, "y_val": y_val,
                           "state": copy.deepcopy(model.state_dict())}, 0
        else:
            stale += 1
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "val_auc": roc_auc_score(y_val, p_val), "best": improved})
        print(f"  [{name} seed {seed}] epoch {epoch:02d} train {train_loss:.4f} val {val_loss:.4f} "
              f"auc {history[-1]['val_auc']:.4f}{' *' if improved else ''}")
        if stale >= cfg["patience"]:
            break
    model.load_state_dict(best.pop("state"))
    return model.eval(), best, pd.DataFrame(history)


def choose_threshold(y, p, target):
    ok = [t for t in np.round(np.arange(0.01, 0.995, 0.005), 3)
          if recall_score(y, (p >= t).astype(int), zero_division=0) >= target]
    return float(max(ok)) if ok else 0.5


def tkey(target):
    return f"{target:.2f}"


def run_base(name, seed):
    return os.path.join(RUN_DIR, f"{name}_seed{seed}")


for seed in FINAL_SEEDS:
    split = make_split(seed)
    for name, cfg in SETUPS.items():
        base = run_base(name, seed)
        if os.path.exists(base + ".json"):
            print(f"skip (done): {name}_seed{seed}")
            continue
        t0 = time.time()
        model, best, history = train_setup(name, seed, split)
        thresholds = {tkey(t): choose_threshold(best["y_val"], best["p_val"], t) for t in TARGET_RECALLS}
        torch.save(model.state_dict(), base + ".pt")
        np.savez(base + "_val_predictions.npz", p_val=best["p_val"], y_val=best["y_val"], rows_val=split["val"])
        history.to_csv(base + "_history.csv", index=False)
        result = {"setup": name, "seed": seed, "lock_id": lock_id, "parameters": count_params(model),
                  "epochs_run": len(history), "best_epoch": best["epoch"], "best_val_loss": best["loss"],
                  "train_minutes": (time.time() - t0) / 60, "thresholds": thresholds,
                  "val_roc_auc": roc_auc_score(best["y_val"], best["p_val"])}
        with open(base + ".json", "w") as f:        # written last: marks the run as complete
            json.dump(result, f, indent=1, default=float)
        print(f"DONE {name}_seed{seed}: thresholds {thresholds} | val AUC {result['val_roc_auc']:.4f} | "
              f"{result['train_minutes']:.1f} min\n")
        del model

## 6. Stage B: freeze

Lists every trained model, its thresholds and a fingerprint of its weight file. Stage C refuses to run unless all runs are present.

In [ ]:
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


frozen = {"system_model": SYSTEM_MODEL, "lock_id": lock_id, "seeds": FINAL_SEEDS, "target_recalls": TARGET_RECALLS,
          "primary_target": PRIMARY_TARGET, "setups": SETUPS, "runs": {}}
missing = []
for name in SETUPS:
    for seed in FINAL_SEEDS:
        base = run_base(name, seed)
        if not os.path.exists(base + ".json"):
            missing.append(f"{name}_seed{seed}")
            continue
        r = json.load(open(base + ".json"))
        frozen["runs"][f"{name}_seed{seed}"] = {"thresholds": r["thresholds"], "weights_md5": md5_file(base + ".pt")}
assert not missing, f"Training is not finished for {missing}. Run Stage A again before opening the test data."
frozen["fingerprint"] = hashlib.md5(json.dumps([frozen["runs"], frozen["setups"]], sort_keys=True, default=str).encode()).hexdigest()[:12]
with open(os.path.join(OUT_DIR, "frozen_models.json"), "w") as f:
    json.dump(frozen, f, indent=1, default=str)
print(f"Frozen {len(frozen['runs'])} runs, fingerprint {frozen['fingerprint']}. System model: {SYSTEM_MODEL}")
print(pd.DataFrame({k: v["thresholds"] for k, v in frozen["runs"].items()}).T.to_string())

## 7. Stage C: open the locked test set and PVSD

Every frozen model scores the 871 locked test clips and the 346 PVSD files. It also re-scores the test clips with the recording level shifted (the loudness check); the probabilities are saved and the frozen thresholds are applied in Stage D. Predictions are saved, so the report cells below can be rerun without opening the test data again.

Each opening is appended to `test_opened_log.json`. If the models change after the first opening, the cell prints a warning, and the paper must say so.

In [ ]:
log_path = os.path.join(OUT_DIR, "test_opened_log.json")
log = json.load(open(log_path)) if os.path.exists(log_path) else []
if log and log[-1]["fingerprint"] != frozen["fingerprint"]:
    print("WARNING: the test data was already opened with different models "
          f"(fingerprint {log[-1]['fingerprint']}). Report this in the paper.")
log.append({"opened_at": datetime.now(timezone.utc).isoformat(), "fingerprint": frozen["fingerprint"],
            "system_model": SYSTEM_MODEL, "quick_run": bool(QUICK_RUN)})
json.dump(log, open(log_path, "w"), indent=1)
print(f"Test data opening #{len(log)} logged.")

LOCKED_SET = set(LOCKED.tolist())


def load_model(name, seed):
    m = build_model(SETUPS[name]["model"], False, SETUPS[name]["head_dropout"])
    m.load_state_dict(torch.load(run_base(name, seed) + ".pt", map_location="cpu"))
    return m.to(device).eval()


def score_clips(model, cfg, indices, allowed, level_offset_db=0.0):
    loader = make_loader(indices, False, 0, allowed=allowed)
    _, p, y = run_epoch(model, loader, cfg, nn.CrossEntropyLoss(), level_offset_db=level_offset_db)
    return p, y


def score_file(model, cfg, clip):
    """Mean Distress probability over non-overlapping 5-s windows of one file."""
    starts = [0] if len(clip) <= CROP_SAMPLES else list(range(0, len(clip) - CROP_SAMPLES + 1, CROP_SAMPLES))
    if len(starts) > MAX_EXTERNAL_WINDOWS:
        keep = np.linspace(0, len(starts) - 1, MAX_EXTERNAL_WINDOWS).round().astype(int)
        starts = [starts[i] for i in keep]
    probs = []
    with torch.no_grad():
        for i in range(0, len(starts), 32):
            segs = []
            for s in starts[i:i + 32]:
                seg = clip[s:s + CROP_SAMPLES].astype(np.float32) / 32768.0
                segs.append((np.pad(seg, (0, CROP_SAMPLES - len(seg))), len(seg)))
            wav = torch.from_numpy(np.stack([a for a, _ in segs])).to(device)
            n = torch.tensor([b for _, b in segs])
            x = normalize(featurizer(wav, n), cfg["norm"])
            probs.append(F.softmax(model(x), dim=1)[:, 1].cpu())
    return torch.cat(probs).mean().item()


for name, cfg in SETUPS.items():
    for seed in FINAL_SEEDS:
        model = load_model(name, seed)
        p_test, y_test = score_clips(model, cfg, LOCKED, LOCKED_SET)
        shifted = {f"p_test_{off:+d}dB": p_test if off == 0 else score_clips(model, cfg, LOCKED, LOCKED_SET, float(off))[0]
                   for off in LEVEL_OFFSETS_DB}
        p_pvsd = np.array([score_file(model, cfg, clips[i]) for i in pvsd_idx])
        np.savez(os.path.join(TEST_DIR, f"{name}_seed{seed}.npz"), p_test=p_test, y_test=y_test, rows_test=LOCKED,
                 p_pvsd=p_pvsd, rows_pvsd=pvsd_idx, **shifted)
        print(f"scored {name}_seed{seed}: test AUC {roc_auc_score(y_test, p_test):.4f}"
              + (f" | PVSD AUC {roc_auc_score(manifest.loc[pvsd_idx, 'label'], p_pvsd):.4f}" if len(pvsd_idx) else ""))
        del model

## 8. Stage D: results on the locked test set

Every number below comes from the saved predictions. Results are given at **two operating points**, fixed before the test data was opened: each run's threshold chosen on its own validation clips to catch **90%** of Distress clips, and to catch **95%**. Results are mean ± standard deviation over the 5 seeds; the test set is the same 871 clips for every seed, so the spread shows how much training varies.

**95% confidence intervals** come from resampling the test **recordings** with replacement (clips of one recording stay together) and recomputing the seed-averaged metric each time.

In [ ]:
def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
            "distress_recall": tp / max(tp + fn, 1), "distress_precision": tp / max(tp + fp, 1),
            "distress_f1": f1_score(y, pred, zero_division=0), "fnr": fn / max(fn + tp, 1),
            "fpr": fp / max(fp + tn, 1), "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p)}


def pm(s, digits=3):
    s = pd.Series(s, dtype=float)
    return f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}" if len(s) > 1 else f"{s.mean():.{digits}f}"


def thr(name, seed, target):
    return frozen["runs"][f"{name}_seed{seed}"]["thresholds"][tkey(target)]


preds = {(n, s): np.load(os.path.join(TEST_DIR, f"{n}_seed{s}.npz")) for n in SETUPS for s in FINAL_SEEDS}
y_test = preds[(SYSTEM_MODEL, FINAL_SEEDS[0])]["y_test"]
test_sources = manifest.loc[LOCKED, "source"].to_numpy()
test_recordings = manifest.loc[LOCKED, "recording"].to_numpy()

rows = []
for t in TARGET_RECALLS:
    for (n, s), pr in preds.items():
        th = thr(n, s, t)
        flagged = pr["p_test"] >= th
        rows.append({"target_recall": t, "setup": n, "seed": s, "threshold": th,
                     **metrics_at(pr["y_test"], pr["p_test"], th),
                     **{f"{k}_at_0.5": v for k, v in metrics_at(pr["y_test"], pr["p_test"], 0.5).items()},
                     "false_alarms_normal": float(flagged[test_sources == "Normal"].mean()),
                     "false_alarms_noise": float(flagged[test_sources == "Noise"].mean())})
test_runs = pd.DataFrame(rows)
test_runs.to_csv(os.path.join(OUT_DIR, "test_results_per_run.csv"), index=False)

# Bootstrap over recordings.
rng = np.random.default_rng(2026)
groups = pd.Series(np.arange(len(test_recordings))).groupby(test_recordings).apply(list).tolist()
boot_metrics = ["accuracy", "distress_recall", "fpr", "roc_auc"]
boot = {(t, n): {m: [] for m in boot_metrics} for t in TARGET_RECALLS for n in SETUPS}
for _ in range(N_BOOTSTRAP):
    pick = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
    yb = y_test[pick]
    if yb.min() == yb.max():
        continue
    for n in SETUPS:
        aucs = [roc_auc_score(yb, preds[(n, s)]["p_test"][pick]) for s in FINAL_SEEDS]
        for t in TARGET_RECALLS:
            vals = {m: [] for m in boot_metrics}
            for s in FINAL_SEEDS:
                flag = preds[(n, s)]["p_test"][pick] >= thr(n, s, t)
                vals["accuracy"].append((flag == yb).mean())
                vals["distress_recall"].append(flag[yb == 1].mean())
                vals["fpr"].append(flag[yb == 0].mean())
            for m in ["accuracy", "distress_recall", "fpr"]:
                boot[(t, n)][m].append(np.mean(vals[m]))
            boot[(t, n)]["roc_auc"].append(np.mean(aucs))
ci = {key: {m: np.percentile(v[m], [2.5, 97.5]) for m in boot_metrics} for key, v in boot.items()}

cols = {"accuracy": "Accuracy", "macro_f1": "Macro-F1", "distress_recall": "Distress recall",
        "distress_precision": "Distress precision", "distress_f1": "Distress F1", "fnr": "FNR", "fpr": "FPR",
        "roc_auc": "ROC-AUC", "pr_auc": "PR-AUC"}
majority = metrics_at(y_test, np.zeros(len(y_test)), 0.5)
main_table = [{"Target recall": "—", "Model": "Majority class (always Non-distress)", "Parameters": "0", "Threshold": "",
               **{v: f"{majority[k]:.3f}" for k, v in cols.items() if k not in ("roc_auc", "pr_auc")}, "ROC-AUC": "0.500"}]
for t in TARGET_RECALLS:
    for n in SETUPS:
        g = test_runs[(test_runs["setup"] == n) & (test_runs["target_recall"] == t)]
        params = json.load(open(run_base(n, FINAL_SEEDS[0]) + ".json"))["parameters"]
        main_table.append({"Target recall": f"{t:.2f}", "Model": LABELS[n], "Parameters": f"{params:,}",
                           "Threshold": pm(g["threshold"]), **{v: pm(g[k]) for k, v in cols.items()}})
main_table = pd.DataFrame(main_table).fillna("")
main_table.to_csv(os.path.join(OUT_DIR, "table_test_main.csv"), index=False)
pd.set_option("display.width", 250)
print(f"LOCKED TEST SET ({len(y_test)} clips: Non-distress {int((y_test == 0).sum())}, Distress {int((y_test == 1).sum())}); "
      f"thresholds chosen on validation; mean ± SD over {len(FINAL_SEEDS)} seeds")
for t in TARGET_RECALLS:
    print(f"\nOperating point: threshold set on validation for {t:.0%} distress recall"
          + (" (primary)" if t == PRIMARY_TARGET else ""))
    print(main_table[main_table["Target recall"].isin(["—", f"{t:.2f}"])].drop(columns="Target recall").to_string(index=False))

ci_table = pd.DataFrame([{"Target recall": f"{t:.2f}", "Model": LABELS[n],
                          **{f"{cols[m]} 95% CI": f"{ci[(t, n)][m][0]:.3f} – {ci[(t, n)][m][1]:.3f}" for m in boot_metrics}}
                         for t in TARGET_RECALLS for n in SETUPS])
ci_table.to_csv(os.path.join(OUT_DIR, "table_test_confidence_intervals.csv"), index=False)
print(f"\n95% confidence intervals of the seed-averaged metrics ({N_BOOTSTRAP} resamples of test recordings):")
print(ci_table.to_string(index=False))

In [ ]:
# What changes for the system model when the target goes from 90% to 95% recall.
sm_rows = []
for t in TARGET_RECALLS:
    g = test_runs[(test_runs["setup"] == SYSTEM_MODEL) & (test_runs["target_recall"] == t)]
    sm_rows.append({"Target recall": f"{t:.0%}", "Threshold": g["threshold"].mean(), "Distress recall": g["distress_recall"].mean(),
                    "Missed distress (FNR)": g["fnr"].mean(), "False-positive rate": g["fpr"].mean(),
                    "Healthy calls flagged": g["false_alarms_normal"].mean(), "Noise flagged": g["false_alarms_noise"].mean(),
                    "Distress precision": g["distress_precision"].mean(), "Accuracy": g["accuracy"].mean()})
target_change = pd.DataFrame(sm_rows).set_index("Target recall")
if len(TARGET_RECALLS) >= 2:
    target_change.loc["change"] = target_change.iloc[-1] - target_change.iloc[0]
target_change.round(3).to_csv(os.path.join(OUT_DIR, "table_test_target_recall_comparison.csv"))
print(f"{LABELS[SYSTEM_MODEL]}: effect of the recall target (mean over seeds, locked test set)")
print(target_change.round(3).T.to_string())

In [ ]:
# The same models at threshold 0.5, and false alarms by source category.
primary_runs = test_runs[test_runs["target_recall"] == PRIMARY_TARGET]
t05 = pd.DataFrame([{"Model": LABELS[n], **{v: pm(primary_runs.loc[primary_runs["setup"] == n, f"{k}_at_0.5"])
                                            for k, v in cols.items() if k not in ("roc_auc", "pr_auc")}}
                    for n in SETUPS])
t05.to_csv(os.path.join(OUT_DIR, "table_test_threshold_0_5.csv"), index=False)
print("Locked test set at threshold 0.5:")
print(t05.to_string(index=False))

by_source = pd.DataFrame([{"Target recall": f"{t:.2f}", "Model": LABELS[n],
                           "Distress clips flagged (recall)": pm(g["distress_recall"]),
                           "Normal clips flagged": pm(g["false_alarms_normal"]), "Noise clips flagged": pm(g["false_alarms_noise"])}
                          for t in TARGET_RECALLS for n in SETUPS
                          for g in [test_runs[(test_runs["setup"] == n) & (test_runs["target_recall"] == t)]]])
by_source.to_csv(os.path.join(OUT_DIR, "table_test_false_alarms_by_source.csv"), index=False)
print("\nShare of test clips flagged as Distress, by source category:")
print(by_source.to_string(index=False))

In [ ]:
# Paired comparison: the system model against each other model, seed by seed (same seed = same training split).
pairs = []
for t in TARGET_RECALLS:
    for other in [n for n in SETUPS if n != SYSTEM_MODEL]:
        a = test_runs[(test_runs["setup"] == SYSTEM_MODEL) & (test_runs["target_recall"] == t)].set_index("seed")
        b = test_runs[(test_runs["setup"] == other) & (test_runs["target_recall"] == t)].set_index("seed")
        metrics = [("distress_f1", 1), ("accuracy", 1), ("fpr", -1), ("distress_recall", 1)]
        if t == PRIMARY_TARGET:
            metrics = [("roc_auc", 1)] + metrics          # ROC-AUC does not depend on the threshold
        for metric, better in metrics:
            d = a[metric] - b[metric]
            pairs.append({"Target recall": f"{t:.2f}", "Compared with": LABELS[other], "Metric": cols[metric],
                          "Mean difference": round(d.mean(), 4), "SD": round(d.std(ddof=1), 4) if len(d) > 1 else None,
                          "Seeds where the system model is better": int(((d * better) > 0).sum())})
pairs = pd.DataFrame(pairs)
pairs.to_csv(os.path.join(OUT_DIR, "table_test_paired_comparison.csv"), index=False)
print(f"{LABELS[SYSTEM_MODEL]} minus each other model (FPR: lower is better):")
print(pairs.to_string(index=False))

# Average of the 5 seed models of each setup (no threshold can be chosen for it on validation, so only
# ROC-AUC and the 0.5 threshold are reported).
ens = []
for n in SETUPS:
    p = np.mean([preds[(n, s)]["p_test"] for s in FINAL_SEEDS], axis=0)
    m = metrics_at(y_test, p, 0.5)
    ens.append({"Model": LABELS[n] + f", average of {len(FINAL_SEEDS)} seeds", "ROC-AUC": round(m["roc_auc"], 4),
                "Accuracy @0.5": round(m["accuracy"], 4), "Distress recall @0.5": round(m["distress_recall"], 4),
                "FPR @0.5": round(m["fpr"], 4)})
ens = pd.DataFrame(ens)
ens.to_csv(os.path.join(OUT_DIR, "table_test_seed_average.csv"), index=False)
print("\nSeed-averaged models (for reference; the gateway runs a single model):")
print(ens.to_string(index=False))

In [ ]:
# Loudness check on the locked test set (each run's threshold held fixed).
lv_rows = []
for t in TARGET_RECALLS:
    for n in SETUPS:
        row = {"Target recall": f"{t:.2f}", "Model": LABELS[n]}
        for off in LEVEL_OFFSETS_DB:
            rec, fa, auc = [], [], []
            for s in FINAL_SEEDS:
                p_off = preds[(n, s)][f"p_test_{off:+d}dB"]
                flag = p_off >= thr(n, s, t)
                rec.append(flag[y_test == 1].mean())
                fa.append(flag[y_test == 0].mean())
                auc.append(roc_auc_score(y_test, p_off))
            row[f"Recall {off:+d} dB"] = round(float(np.mean(rec)), 3)
            row[f"FPR {off:+d} dB"] = round(float(np.mean(fa)), 3)
            row[f"ROC-AUC {off:+d} dB"] = round(float(np.mean(auc)), 3)
        lv_rows.append(row)
level_table = pd.DataFrame(lv_rows)
level_table.to_csv(os.path.join(OUT_DIR, "table_test_loudness_check.csv"), index=False)
for label in ["Recall", "FPR", "ROC-AUC"]:
    print(f"{label} on the locked test set with the recording level shifted (mean over seeds):")
    print(level_table[["Target recall", "Model"] + [f"{label} {off:+d} dB" for off in LEVEL_OFFSETS_DB]].to_string(index=False), "\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for t in TARGET_RECALLS:
    for n in SETUPS:
        if n != SYSTEM_MODEL and t != PRIMARY_TARGET:
            continue
        r = level_table[(level_table["Model"] == LABELS[n]) & (level_table["Target recall"] == f"{t:.2f}")].iloc[0]
        style = "-" if t == PRIMARY_TARGET else "--"
        label = f"{LABELS[n]}, {t:.0%} target"
        axes[0].plot(LEVEL_OFFSETS_DB, [r[f"Recall {o:+d} dB"] for o in LEVEL_OFFSETS_DB], style, marker="o", label=label)
        axes[1].plot(LEVEL_OFFSETS_DB, [r[f"FPR {o:+d} dB"] for o in LEVEL_OFFSETS_DB], style, marker="o", label=label)
axes[0].set_ylabel("Distress recall")
axes[1].set_ylabel("False-positive rate")
for ax in axes:
    ax.set_xlabel("Recording level shift (dB)")
    ax.axvline(0, color="gray", linewidth=0.8)
axes[0].legend(fontsize=7)
plt.suptitle("Locked test set, thresholds fixed at their validation values")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_loudness_check.png"), dpi=200)
plt.show()

In [ ]:
# ROC curves, and the system model's confusion matrix at each operating point (primary seed).
fig, axes = plt.subplots(1, 1 + len(TARGET_RECALLS), figsize=(5.5 * (1 + len(TARGET_RECALLS)), 4.5))
for n in SETUPS:
    pr = preds[(n, PRIMARY_SEED)]
    fpr, tpr, _ = roc_curve(pr["y_test"], pr["p_test"])
    axes[0].plot(fpr, tpr, label=f"{LABELS[n]} (AUC {roc_auc_score(pr['y_test'], pr['p_test']):.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
axes[0].set_xlabel("False-positive rate")
axes[0].set_ylabel("True-positive rate (distress recall)")
axes[0].set_title(f"ROC curves, locked test set, seed {PRIMARY_SEED}")
axes[0].legend(fontsize=7)
pr = preds[(SYSTEM_MODEL, PRIMARY_SEED)]
for ax, t in zip(axes[1:], TARGET_RECALLS):
    th = thr(SYSTEM_MODEL, PRIMARY_SEED, t)
    cm = confusion_matrix(pr["y_test"], (pr["p_test"] >= th).astype(int), labels=[0, 1])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(CLASS_NAMES, rotation=90, va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{LABELS[SYSTEM_MODEL]}\n{t:.0%} target, threshold {th:.3f}, seed {PRIMARY_SEED}", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "figure_test_roc_and_confusion.png"), dpi=200)
plt.show()

## 9. Results on PVSD (external test)

Per file, at each run's validation thresholds and at 0.5. PVSD labels describe a treated versus untreated group of birds, not individual clips, and PVSD was recorded with different equipment. Read these results as a stress test of transfer to new recordings, not as a measure of disease detection.

In [ ]:
if len(pvsd_idx) == 0:
    print("PVSD not found; skipped.")
else:
    y_pvsd = manifest.loc[pvsd_idx, "label"].to_numpy()
    pv_rows = []
    for t in TARGET_RECALLS:
        for (n, s), pr in preds.items():
            pv_rows.append({"target_recall": t, "setup": n, "seed": s, **metrics_at(y_pvsd, pr["p_pvsd"], thr(n, s, t)),
                            **{f"{k}_at_0.5": v for k, v in metrics_at(y_pvsd, pr["p_pvsd"], 0.5).items()}})
    pv_runs = pd.DataFrame(pv_rows)
    pv_runs.to_csv(os.path.join(OUT_DIR, "pvsd_results_per_run.csv"), index=False)
    pv_table = pd.DataFrame([{"Target recall": f"{t:.2f}", "Model": LABELS[n],
                              **{v: pm(g[k]) for k, v in [("accuracy", "Accuracy"), ("distress_recall", "Distress recall"),
                                                          ("fpr", "FPR"), ("roc_auc", "ROC-AUC"), ("pr_auc", "PR-AUC")]},
                              "Accuracy @0.5": pm(g["accuracy_at_0.5"])}
                             for t in TARGET_RECALLS for n in SETUPS
                             for g in [pv_runs[(pv_runs["setup"] == n) & (pv_runs["target_recall"] == t)]]])
    pv_table.to_csv(os.path.join(OUT_DIR, "table_pvsd.csv"), index=False)
    print(f"PVSD: {len(y_pvsd)} files (Non-distress {int((y_pvsd == 0).sum())}, Distress {int((y_pvsd == 1).sum())}); "
          f"always predicting Non-distress scores {max((y_pvsd == 0).mean(), (y_pvsd == 1).mean()):.3f} accuracy")
    print(pv_table.to_string(index=False))

    pr = preds[(SYSTEM_MODEL, PRIMARY_SEED)]
    pv = manifest.loc[pvsd_idx, ["file", "original_label", "seconds"]].copy()
    pv["prob_distress"] = pr["p_pvsd"]
    for t in TARGET_RECALLS:
        pv[f"flagged_{tkey(t)}"] = pr["p_pvsd"] >= thr(SYSTEM_MODEL, PRIMARY_SEED, t)
    pv.to_csv(os.path.join(OUT_DIR, "pvsd_predictions_system_model.csv"), index=False)
    print(f"\n{LABELS[SYSTEM_MODEL]}, seed {PRIMARY_SEED}: share of PVSD files flagged, by original label")
    print(pv.groupby("original_label")[[f"flagged_{tkey(t)}" for t in TARGET_RECALLS]].mean().round(3).to_string())
    short = pv["seconds"] < CROP_SEC
    flag_col = f"flagged_{tkey(PRIMARY_TARGET)}"
    print(f"Files shorter than {CROP_SEC:.0f} s: {int(short.sum())}, flagged {pv.loc[short, flag_col].mean():.3f}; "
          f"longer files flagged {pv.loc[~short, flag_col].mean():.3f} (primary target)")

## 10. Stage E: zone-level alerts

The gateway raises an alert for a zone when at least a set share of that zone's most recent clips is flagged. This part chooses that rule, once for each operating point, and reports how it behaves.

**Simulation.** One 5-s clip arrives per zone every 30 seconds. A healthy zone's clips are drawn from Non-distress clips; a sick zone's clips contain distress with a given probability (the outbreak share). Each drawn clip is flagged or not according to how the system model actually classified a real clip of that kind.

**False alerts** are counted as *alert-hours*: the number of hours in a week during which a healthy zone shows an alert at any moment. An alert that switches on and stays on all week therefore counts as 168, not as 1.

1. **Choosing the rule (validation only).** Using the system model's validation predictions from all seeds, every combination of window length and share is simulated. The chosen rule is the one with the shortest median time to alert at an outbreak share of 0.3, among rules that give at most one false alert-hour per healthy zone per week.
2. **Reporting (test).** The chosen rule is then simulated with the locked-test predictions.

**Limitation.** Clips are drawn independently, but in a real poultry house consecutive clips are related (the same birds and the same noise), so errors cluster. Real false-alert rates are likely higher than simulated ones. The controlled playback test and field observation are still needed.

In [ ]:
def flag_pools(kind, target):
    """Flags (True/False) of Non-distress and Distress clips for the system model, all seeds pooled."""
    neg, pos = [], []
    for s in FINAL_SEEDS:
        th = thr(SYSTEM_MODEL, s, target)
        if kind == "validation":
            v = np.load(run_base(SYSTEM_MODEL, s) + "_val_predictions.npz")
            p, y = v["p_val"], v["y_val"]
        else:
            p, y = preds[(SYSTEM_MODEL, s)]["p_test"], preds[(SYSTEM_MODEL, s)]["y_test"]
        neg.append(p[y == 0] >= th)
        pos.append(p[y == 1] >= th)
    return np.concatenate(neg), np.concatenate(pos)


def alert_on(flags, window, rate):
    """True at each clip where at least `rate` of the last `window` clips are flagged."""
    need = math.ceil(rate * window)
    counts = np.convolve(flags.astype(int), np.ones(window, dtype=int), mode="full")[:len(flags)]
    on = counts >= need
    on[:window - 1] = False
    return on


def simulate(neg, pos, window, rate, rng, reps=ZONE_REPS):
    per_hour = int(60 / CLIP_INTERVAL_MIN)
    week = 7 * 24 * per_hour
    alert_hours = [alert_on(rng.choice(neg, week), window, rate).reshape(-1, per_hour).any(axis=1).sum()
                   for _ in range(reps)]
    out = {"false_alert_hours_per_week": float(np.mean(alert_hours))}
    pre, post = 2 * per_hour, 24 * per_hour             # 2 healthy hours, then up to 24 hours of outbreak
    for share in OUTBREAK_SHARES:
        delays = []
        for _ in range(reps):
            sick = rng.random(post) < share
            stream = np.concatenate([rng.choice(neg, pre), np.where(sick, rng.choice(pos, post), rng.choice(neg, post))])
            on = np.flatnonzero(alert_on(stream, window, rate)[pre:])
            delays.append(on[0] * CLIP_INTERVAL_MIN if len(on) else np.inf)
        delays = np.array(delays)
        out[f"detected_within_24h_share_{share}"] = float(np.isfinite(delays).mean())
        out[f"median_minutes_to_alert_share_{share}"] = float(np.median(delays))
    return out


RULES, zone_results, zone_rows = {}, {}, []
key = f"median_minutes_to_alert_share_{DESIGN_SHARE}"
for t in TARGET_RECALLS:
    rng = np.random.default_rng(2026)
    neg_val, pos_val = flag_pools("validation", t)
    grid = pd.DataFrame([{"window_clips": w, "share_flagged": r, **simulate(neg_val, pos_val, w, r, rng)}
                         for w in ALERT_WINDOWS for r in ALERT_RATES])
    grid.to_csv(os.path.join(OUT_DIR, f"zone_rule_search_validation_{tkey(t)}.csv"), index=False)
    ok = grid[grid["false_alert_hours_per_week"] <= MAX_FALSE_ALERT_HOURS_PER_WEEK]
    if len(ok):
        chosen = ok.sort_values([key, "window_clips"]).iloc[0]
    else:
        print(f"WARNING ({t:.0%} target): no rule meets the false-alert limit on validation. Using the rule with the "
              "fewest false alert-hours instead; report this in the paper.")
        chosen = grid.sort_values(["false_alert_hours_per_week", key]).iloc[0]
    RULES[t] = {"window_clips": int(chosen["window_clips"]), "share_flagged": float(chosen["share_flagged"])}
    neg_test, pos_test = flag_pools("test", t)
    zone_results[t] = simulate(neg_test, pos_test, RULES[t]["window_clips"], RULES[t]["share_flagged"], np.random.default_rng(7))
    print(f"{t:.0%} target. Validation clip-level FPR {neg_val.mean():.3f}, recall {pos_val.mean():.3f}. "
          f"Chosen rule: at least {math.ceil(RULES[t]['share_flagged'] * RULES[t]['window_clips'])} of the last "
          f"{RULES[t]['window_clips']} clips ({RULES[t]['window_clips'] * CLIP_INTERVAL_MIN:.0f} minutes) flagged. "
          f"Test clip-level FPR {neg_test.mean():.3f}, recall {pos_test.mean():.3f}.")
    for s in OUTBREAK_SHARES:
        zone_rows.append({"Target recall": f"{t:.0%}", "Rule": f"{math.ceil(RULES[t]['share_flagged'] * RULES[t]['window_clips'])} of "
                                                               f"{RULES[t]['window_clips']} clips",
                          "False alert-hours per week": round(zone_results[t]["false_alert_hours_per_week"], 2),
                          "Outbreak share": s, "Detected within 24 h": zone_results[t][f"detected_within_24h_share_{s}"],
                          "Median minutes to alert": zone_results[t][f"median_minutes_to_alert_share_{s}"]})
zone_table = pd.DataFrame(zone_rows)
zone_table.to_csv(os.path.join(OUT_DIR, "table_zone_alerts_test.csv"), index=False)
json.dump({tkey(t): {"rule": RULES[t], **zone_results[t]} for t in TARGET_RECALLS},
          open(os.path.join(OUT_DIR, "zone_alerts_test.json"), "w"), indent=1, default=float)
print("\nZone alerts on locked-test predictions (simulation):")
print(zone_table.round(3).to_string(index=False))

## 11. Stage F: export the system model for the Raspberry Pi

Three files go to `OUT_DIR/export`:

- `sonara_system_model.pt`: a TorchScript file that takes one 5-second, 16 kHz mono clip (shape 1 × 80000, values in −1…1) and returns the Distress probability. It includes the feature extraction, so the gateway needs only PyTorch.
- `mobilenetv3_small_seed42.pt`: the plain weights, in the format `test/pi_latency_benchmark.py --weights_dir` expects.
- `deployment_config.json`: the thresholds and alert rules for both operating points, the primary one marked, and the input format.

In [ ]:
class DeployModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.featurizer = LogMelDeltas()
        self.model = model

    def forward(self, wav):
        x = self.featurizer.features(wav)[..., :N_FRAMES]
        return F.softmax(self.model(x), dim=1)[:, 1]


assert SETUPS[SYSTEM_MODEL]["norm"] == "none", "the export assumes no normalization"
system = load_model(SYSTEM_MODEL, PRIMARY_SEED).cpu().eval()
deploy = DeployModel(system).eval()
example = torch.from_numpy(clips[int(LOCKED[0])][:CROP_SAMPLES].astype(np.float32) / 32768.0).unsqueeze(0)
with torch.no_grad():
    scripted = torch.jit.trace(deploy, example)
export_dir = os.path.join(OUT_DIR, "export")
scripted.save(os.path.join(export_dir, "sonara_system_model.pt"))
torch.save(system.state_dict(), os.path.join(export_dir, f"{SETUPS[SYSTEM_MODEL]['model']}_seed{PRIMARY_SEED}.pt"))
config = {"setup": SYSTEM_MODEL, "seed": PRIMARY_SEED, "primary_target_recall": PRIMARY_TARGET,
          "operating_points": {tkey(t): {"threshold": thr(SYSTEM_MODEL, PRIMARY_SEED, t),
                                         "alert_rule": {**RULES[t], "clip_interval_minutes": CLIP_INTERVAL_MIN}}
                               for t in TARGET_RECALLS},
          "input": {"sample_rate": SAMPLE_RATE, "seconds": CROP_SEC, "channels": 1, "range": "-1 to 1",
                    "shorter_clips": "pad with zeros to 80000 samples"},
          "frozen_fingerprint": frozen["fingerprint"]}
json.dump(config, open(os.path.join(export_dir, "deployment_config.json"), "w"), indent=1)

# Check that the exported file gives the same answer as the trained model.
reloaded = torch.jit.load(os.path.join(export_dir, "sonara_system_model.pt"))
check = []
with torch.no_grad():
    for i in LOCKED[:20]:
        wav = torch.from_numpy(clips[int(i)][:CROP_SAMPLES].astype(np.float32) / 32768.0)
        wav = F.pad(wav, (0, CROP_SAMPLES - len(wav))).unsqueeze(0)
        check.append(abs(reloaded(wav).item() - deploy(wav).item()))
print(f"Exported to {export_dir}; largest difference from the trained model on 20 test clips: {max(check):.2e}")
print(json.dumps(config, indent=1))

## 12. Values for the paper

The cell below collects the numbers for the results chapter. Report the test-set table as the main result with its confidence intervals, both operating points, PVSD as an external stress test, and the zone results as a simulation.

In [ ]:
print(f"System model: {LABELS[SYSTEM_MODEL]}, {len(FINAL_SEEDS)} seeds, locked test set of {len(y_test)} clips (id {lock_id}).")
for t in TARGET_RECALLS:
    sm = test_runs[(test_runs["setup"] == SYSTEM_MODEL) & (test_runs["target_recall"] == t)]
    base = test_runs[(test_runs["setup"] == "A0_cnn_bilstm_run1") & (test_runs["target_recall"] == t)]
    c = ci[(t, SYSTEM_MODEL)]
    print(f"\n{t:.0%} recall target{' (primary)' if t == PRIMARY_TARGET else ''}, mean threshold {sm['threshold'].mean():.3f}:")
    print(f"  accuracy {pm(sm['accuracy'])} (95% CI {c['accuracy'][0]:.3f}–{c['accuracy'][1]:.3f}), "
          f"distress recall {pm(sm['distress_recall'])} ({c['distress_recall'][0]:.3f}–{c['distress_recall'][1]:.3f}), "
          f"false-positive rate {pm(sm['fpr'])} ({c['fpr'][0]:.3f}–{c['fpr'][1]:.3f}), "
          f"ROC-AUC {pm(sm['roc_auc'])} ({c['roc_auc'][0]:.3f}–{c['roc_auc'][1]:.3f}).")
    print(f"  Run 1 CNN-BiLSTM on the same test set: accuracy {pm(base['accuracy'])}, recall {pm(base['distress_recall'])}, "
          f"false-positive rate {pm(base['fpr'])}.")
    z = zone_results[t]
    print(f"  Zone alerts (simulation): rule {RULES[t]}; false alert-hours per healthy zone per week "
          f"{z['false_alert_hours_per_week']:.2f}; median minutes to alert at outbreak share {DESIGN_SHARE}: {z[key]:.1f}.")
sm0 = test_runs[(test_runs["setup"] == SYSTEM_MODEL) & (test_runs["target_recall"] == PRIMARY_TARGET)]
print(f"\nAt threshold 0.5: accuracy {pm(sm0['accuracy_at_0.5'])}, distress F1 {pm(sm0['distress_f1_at_0.5'])}.")
print(f"Test data openings logged: {len(json.load(open(log_path)))}.")